In [195]:
from   tensorflow.keras.models import Sequential,Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from collections import deque
from sklearn import preprocessing
import random

In [196]:
main_data=pd.DataFrame()

In [197]:
currency=['BCH-USD','BTC-USD','ETH-USD','LTC-USD']
for  c in currency:
    dataset='Digital_Currency_data/'+c+'.csv'
    df=pd.read_csv(dataset,names=['time','Low','High','open','Close','Volume'])
    df.rename(columns={'Close':c+'_Close','Volume':c+'_Volume'},inplace=True)
    df.set_index('time',inplace=True)
    df=df[[c+'_Close',c+'_Volume']]
    if len(main_data)==0:
        main_data=df
    else:
        main_data=main_data.join(df)

In [198]:
main_data.head()

,BCH-USD_Close,BCH-USD_Volume,BTC-USD_Close,BTC-USD_Volume,ETH-USD_Close,ETH-USD_Volume,LTC-USD_Close,LTC-USD_Volume
time,,,,,,,,
1528968660,871.719971,5.675361,6489.549805,0.587100,NaN,NaN,96.580002,9.647200
1528968720,870.859985,26.856577,6487.379883,7.706374,486.01001,26.019083,96.660004,314.387024
1528968780,870.099976,1.124300,6479.410156,3.088252,486.00000,8.449400,96.570000,77.129799
1528968840,870.789978,1.749862,6479.410156,1.404100,485.75000,26.994646,96.500000,7.216067
1528968900,870.000000,1.680500,6479.979980,0.753000,486.00000,77.355759,96.389999,524.539978


In [199]:
main_data.isnull().sum()

BCH-USD_Close        0
BCH-USD_Volume       0
BTC-USD_Close     5122
BTC-USD_Volume    5122
ETH-USD_Close      195
ETH-USD_Volume     195
LTC-USD_Close      836
LTC-USD_Volume     836
dtype: int64

In [200]:
main_data.fillna(method='ffill',inplace=True)

C:\Users\LOQ\AppData\Local\Temp\ipykernel_12060\407536374.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  main_data.fillna(method='ffill',inplace=True)


In [201]:
main_data.isnull().sum()

BCH-USD_Close     0
BCH-USD_Volume    0
BTC-USD_Close     0
BTC-USD_Volume    0
ETH-USD_Close     1
ETH-USD_Volume    1
LTC-USD_Close     0
LTC-USD_Volume    0
dtype: int64

In [202]:
main_data.fillna(method='bfill',inplace=True)

C:\Users\LOQ\AppData\Local\Temp\ipykernel_12060\1945591578.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  main_data.fillna(method='bfill',inplace=True)


In [203]:
main_data.isnull().sum()


BCH-USD_Close     0
BCH-USD_Volume    0
BTC-USD_Close     0
BTC-USD_Volume    0
ETH-USD_Close     0
ETH-USD_Volume    0
LTC-USD_Close     0
LTC-USD_Volume    0
dtype: int64

In [204]:
main_data['future']=main_data['LTC-USD_Close'].shift(-3)

In [205]:
main_data.head()

,BCH-USD_Close,BCH-USD_Volume,BTC-USD_Close,BTC-USD_Volume,ETH-USD_Close,ETH-USD_Volume,LTC-USD_Close,LTC-USD_Volume,future
time,,,,,,,,,
1528968660,871.719971,5.675361,6489.549805,0.587100,486.01001,26.019083,96.580002,9.647200,96.500000
1528968720,870.859985,26.856577,6487.379883,7.706374,486.01001,26.019083,96.660004,314.387024,96.389999
1528968780,870.099976,1.124300,6479.410156,3.088252,486.00000,8.449400,96.570000,77.129799,96.519997
1528968840,870.789978,1.749862,6479.410156,1.404100,485.75000,26.994646,96.500000,7.216067,96.440002
1528968900,870.000000,1.680500,6479.979980,0.753000,486.00000,77.355759,96.389999,524.539978,96.470001


In [206]:
def compare(current,future):
    if current>future:
        return 1
    else:
        return 0
main_data['taget']=list(map(compare,main_data['LTC-USD_Close'],main_data['future']))

In [207]:
main_data.head()

,BCH-USD_Close,BCH-USD_Volume,BTC-USD_Close,BTC-USD_Volume,ETH-USD_Close,ETH-USD_Volume,LTC-USD_Close,LTC-USD_Volume,future,taget
time,,,,,,,,,,
1528968660,871.719971,5.675361,6489.549805,0.587100,486.01001,26.019083,96.580002,9.647200,96.500000,1
1528968720,870.859985,26.856577,6487.379883,7.706374,486.01001,26.019083,96.660004,314.387024,96.389999,1
1528968780,870.099976,1.124300,6479.410156,3.088252,486.00000,8.449400,96.570000,77.129799,96.519997,1
1528968840,870.789978,1.749862,6479.410156,1.404100,485.75000,26.994646,96.500000,7.216067,96.440002,1
1528968900,870.000000,1.680500,6479.979980,0.753000,486.00000,77.355759,96.389999,524.539978,96.470001,0


In [208]:
times=sorted(main_data.index.values)

In [209]:
len(times)

92225

In [210]:
int(0.1*len(times))

9222

In [211]:
last_10pct=sorted(main_data.index.values)[-int(0.1*len(times))]

In [212]:
last_10pct

np.int64(1534556340)

In [213]:
main_data_test=main_data[(main_data.index>=last_10pct)]
main_data_train=main_data[(main_data.index<last_10pct)]

In [214]:
len(main_data_test)

9222

In [215]:
len(main_data_train)


83003

In [216]:
main_data['BTC-USD_Close']


time
1528968660    6489.549805
1528968720    6487.379883
1528968780    6479.410156
1528968840    6479.410156
1528968900    6479.979980
                 ...     
1535215020    6714.520020
1535215080    6714.520020
1535215140    6715.000000
1535215200    6715.000000
1535215260    6715.000000
Name: BTC-USD_Close, Length: 92225, dtype: float64

In [217]:
main_data['BTC-USD_Close'].pct_change()

time
1528968660         NaN
1528968720   -0.000334
1528968780   -0.001228
1528968840    0.000000
1528968900    0.000088
                ...   
1535215020    0.000206
1535215080    0.000000
1535215140    0.000071
1535215200    0.000000
1535215260    0.000000
Name: BTC-USD_Close, Length: 92225, dtype: float64

In [218]:
def preprocess(df):
    df=df.drop('future',axis=1)
    for col in df.columns:
        if col!='target':
            df[col]=df[col].pct_change()
            df.dropna(inplace=True)
            df[col]=preprocessing.scale(df[col].values)
            df.dropna(inplace=True)
        squneces=[]
        prev_d=deque(maxlen=30)
        for i in df.values:
             prev_d.append([n for n in i[:]])
             if len(prev_d)==30:
                 squneces.append([np.array(prev_d),i[-1]])
        random.shuffle(squneces)
        buys=[]
        sells=[]
        for sq,target in squneces:
            if target==0:
                sells.append([sq,target])
            elif target==1:
                buys.append([sq,target])
        random.shuffle(buys)
        random.shuffle(sells)
        lower=min(len(buys),len(sells))
        buys=buys[:lower]
        sells=sells[:lower]
        seq_data=buys+sells
        random.shuffle(seq_data)
        x=[]
        y=[]
        for  seg,target in seq_data:
            x.append(seg)
            y.append(target)
        return np.array(x),np.array(y)

In [219]:
train_x,train_y=preprocess(main_data_train)

In [220]:
train_x.shape

(72318, 30, 9)

In [221]:
train_y.shape

(72318,)

In [222]:
test_x,test_y=preprocess(main_data_test)


In [223]:
test_x.shape

(7580, 30, 9)

In [224]:
np.unique(train_y,return_counts=True)

(array([0., 1.]), array([36159, 36159]))

In [225]:
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.layers import Dense,LSTM,Dropout,BatchNormalization

In [226]:
model=Sequential()
model.add(LSTM(64,input_shape=(30,9)))
model.add(Dense(32,activation='relu'))
model.add(Dense(2,activation='softmax'))

C:\Users\LOQ\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [227]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='Adam',metrics=['accuracy'])

In [228]:
history=model.fit(train_x,train_y,batch_size=100,epochs=100,validation_data=(test_x,test_y))

Epoch 1/100
724/724 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.4993 - loss: 0.6943 - val_accuracy: 0.5191 - val_loss: 0.6916
Epoch 2/100
724/724 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.5005 - loss: 0.6938 - val_accuracy: 0.5170 - val_loss: 0.6916
Epoch 3/100
724/724 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.5016 - loss: 0.6935 - val_accuracy: 0.4996 - val_loss: 0.6935
Epoch 4/100
724/724 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.4971 - loss: 0.6938 - val_accuracy: 0.5182 - val_loss: 0.6919
Epoch 5/100
724/724 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.5040 - loss: 0.6938 - val_accuracy: 0.5009 - val_loss: 0.6932
Epoch 6/100
724/724 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.4998 - loss: 0.6934 - val_accuracy: 0.5003 - val_loss: 0.6923
Epoch 7/100
724/724 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4989 - loss: 0.6936 - val_accuracy: 0.5008 - val_loss: 0.6923
Epoch 8/100
724/724 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.4993 - loss: 0.6935 - val_ac